# Stage 04 Homework: Data Acquisition & Ingestion
- API Ingestion with secrets in `.env`
- Scrape public table using BeautifulSoup
- Data validation and automated export to `data/raw/`

In [19]:
# !pip install pandas requests yfinance python-dotenv beautifulsoup4

In [20]:
from pathlib import Path

ROOT = Path.cwd()
CHECKS = [
    (".env", "NEEDED", "Local environment file with secrets"),
    (".env.example", "NEEDED", "Template repository environment file"),
]

print(f"Checking directory: {ROOT}\n")
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<20}  {note}")

Checking directory: d:\NYU Bootcamp\bootcamp_Ziyu_Li\homework\homework04

  [OK ]  NEEDED    .env                  Local environment file with secrets
  [OK ]  NEEDED    .env.example          Template repository environment file


In [21]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# Ensure data/raw directory exists
RAW = pathlib.Path('data/raw')
RAW.mkdir(parents=True, exist_ok=True)

# Load environment variables from .env file
load_dotenv()
print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

# Generate formatted timestamp string for filenames
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# Export DataFrame to CSV with dynamically formatted metadata filename
def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k, v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved raw dataset to:', path)
    return path

# Perform basic quality control & validation on DataFrames
def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {
        'missing_columns': missing,
        'shape': df.shape,
        'na_total': int(df.isna().sum().sum()),
        'dtypes': df.dtypes.to_dict()
    }

ALPHAVANTAGE_API_KEY loaded? True


In [17]:
import os, requests
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('ALPHAVANTAGE_API_KEY')
print("Current Key in Memory:", api_key)

url = 'https://www.alphavantage.co/query'
params = {'function': 'TIME_SERIES_DAILY', 'symbol': 'AAPL', 'apikey': api_key}
res = requests.get(url, params=params).json()
print("\nRaw API Response:\n", res)

Current Key in Memory: K78XYGCSNXA4G2X2

Raw API Response:
 {'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'AAPL', '3. Last Refreshed': '2026-08-31', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}, 'Time Series (Daily)': {'2026-08-31': {'1. open': '319.6000', '2. high': '321.2350', '3. low': '312.8000', '4. close': '316.8500', '5. volume': '41242724'}, '2026-08-28': {'1. open': '316.8450', '2. high': '322.3700', '3. low': '315.4504', '4. close': '319.7000', '5. volume': '38649398'}, '2026-08-27': {'1. open': '310.5450', '2. high': '315.4000', '3. low': '309.4001', '4. close': '314.5800', '5. volume': '32419233'}, '2026-08-26': {'1. open': '310.3000', '2. high': '315.4300', '3. low': '308.8001', '4. close': '313.4500', '5. volume': '34024486'}, '2026-08-25': {'1. open': '310.7900', '2. high': '313.5900', '3. low': '308.2100', '4. close': '309.9000', '5. volume': '25869807'}, '2026-08-24': {'1. open': '311.4700', '2. high': 

In [22]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY')) and os.getenv('ALPHAVANTAGE_API_KEY') != 'your_api_key_here'

# Request daily price series from Alpha Vantage
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function': 'TIME_SERIES_DAILY', 'symbol': SYMBOL, 'outputsize': 'compact', 'apikey': os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage cap reached or error response:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

# Parse response from Alpha Vantage API
if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index': 'date', '4. close': 'close'})[['date', 'close']]
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

# Fallback mechanism using yfinance if API key is missing or rate limited
if not USE_ALPHA:
    import yfinance as yf
    print('Using yfinance fallback...')
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False, multi_level_index=False).reset_index()[['Date', 'Close']]
    df_api.columns = ['date', 'close']
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

# Validate dataset structure
v_api = validate(df_api, ['date', 'close'])
print("API Data Validation Results:", v_api)

# Export raw API dataset to CSV
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

API Data Validation Results: {'missing_columns': [], 'shape': (100, 2), 'na_total': 0, 'dtypes': {'date': dtype('<M8[ns]'), 'close': dtype('float64')}}
Saved raw dataset to: data\raw\api_source-alpha_symbol-AAPL_20260901-144631.csv


In [4]:
# Target public table on Wikipedia (DJIA Components)
SCRAPE_URL = 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Locate table elements with class 'wikitable'
    table = soup.find('table', {'class': 'wikitable'})
    rows = []
    for tr in table.find_all('tr'):
        cols = [c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
        if cols:
            rows.append(cols)
            
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scraping failed; switching to inline fallback table:', e)
    html = '<table><tr><th>Company</th><th>Symbol</th><th>Weight</th></tr><tr><td>Apple</td><td>AAPL</td><td>12.5</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th', 'td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

# Standardize column headers and run validation
df_scrape.columns = [c.replace(' ', '_').lower() for c in df_scrape.columns]
v_scrape = validate(df_scrape, list(df_scrape.columns))
print("Scrape Validation Results:", v_scrape)

# Export scraped dataset to CSV
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='djia_components')

Scrape Validation Results: {'missing_columns': [], 'shape': (130, 4), 'na_total': 0, 'dtypes': {'year': dtype('O'), 'closingvalue': dtype('O'), 'netchange': dtype('O'), 'percentagechange': dtype('O')}}
Saved raw dataset to: data\raw\scrape_site-wikipedia_table-djia_components_20260901-141901.csv
